# PICID experiment: ragged CSTR prognostics

This notebook owns the experiment workflow. The main PICID interface calls—datasource composition, preprocessing, task definition, training, and testing—remain visible and editable here.

Only reusable details are external:

- `cstr_simulator.py`: reactor dynamics, degradation, schedules, failure detection, and PICID-facing data export.
- `picid_training.py`: path setup, transforms, inspection, and result packaging.
- `picid_plotting.py`: raw-data and run-trace plotting.

In [ ]:
from random import Random

from IPython.display import Image, display
from omegaconf import OmegaConf
import pandas as pd
from rich.console import Console
import torch
from torch import nn

from cstr_simulator import CSTR_FEATURE_COLUMNS, CSTRParameters, simulate_cstr_fleet

# Import this first: it configures PICID paths before PICID reads its settings.
from picid_training import (
    DEEP_LEARNING_EPOCHS,
    N_ESTIMATORS,
    SEED,
    SEQ_LEN,
    STRIDE,
    WORKSHOP_ROOT,
    build_checkpoint_unroll,
    execution_comparison_table,
    make_trace_version,
    package_training_run,
    scaling_transforms,
    split_alignment_table,
    summarize_prepared_experiment,
)
from picid_plotting import save_run_traces, save_unit_data_trace

from picid.data.preprocessing import BySourceSplitter
from picid.interface import (
    CustomMultiSourceLoader,
    CustomSingleSourceLoader,
    PicidExperiment,
)
from picid.interface.schemas.loggers import CsvLogger
from picid.interface.schemas.task_definition import Prognostic
from picid.interface.model.custom_model import CustomModelTrainer
from picid.transforms.base import DataTransform
from picid.transforms.base_transforms.reshaping import ReshapeTransform
from picid.transforms.base_transforms.tabularizers import TimeseriesTabularizer

ASSET_DIR = WORKSHOP_ROOT / "assets"
ARTIFACT_DIR = WORKSHOP_ROOT / "artifacts" / "training_monitoring"
N_FEATURES = len(CSTR_FEATURE_COLUMNS)
TRACE_VERSION = make_trace_version(
    seed=SEED,
    deep_learning_epochs=DEEP_LEARNING_EPOCHS,
)
DATA_DIR = WORKSHOP_ROOT / "data" / TRACE_VERSION
CHECKPOINT_DIR = WORKSHOP_ROOT / "artifacts" / "checkpoints" / TRACE_VERSION
Console().print(
    f"seed={SEED}, MLP epochs={DEEP_LEARNING_EPOCHS}, "
    f"trees={N_ESTIMATORS}, fit/predict=BayesianRidge, "
    f"logger version={TRACE_VERSION}"
)

## 1. Generate and inspect the raw units

One high-level simulator call creates 30 seeded CSTR run-to-failure units and persists one PICID-facing CSV per reactor. Catalyst loss is deliberately dominant while thermal fouling remains active, and longer operating regimes expose a transferable multivariate degradation signature without making a fixed threshold reliable across units. Their lengths emerge until a hidden virtual reference-capacity test fails. PICID receives only reactor/controller measurements and known operating context; outlet concentration, conversion, both health states, and the reference test remain privileged diagnostics. See `00_cstr_degradation_simulator.ipynb` for the equations, reactor schema, threshold audit, diagnostic ground truth, and CSV export contract.

In [ ]:
parameters = CSTRParameters(
    deactivation_rate_at_reference=0.009,
    fouling_rate_at_reference=0.003,
    regime_min_steps=80,
    regime_max_steps=120,
)
fleet = simulate_cstr_fleet(n_units=30, seed=SEED, parameters=parameters)
unit_csv_paths = fleet.export_model_csvs(DATA_DIR)
unit_frames = {
    unit_name: pd.read_csv(csv_path, dtype="float32")
    for unit_name, csv_path in unit_csv_paths.items()
}
unit_summary = fleet.summary()
unit_names = list(unit_frames)
split_order = unit_names.copy()
Random(SEED).shuffle(split_order)
n_train = int(0.6 * len(split_order))
n_val = int(0.2 * len(split_order))
unit_names_by_split = {
    "train": split_order[:n_train],
    "val": split_order[n_train : n_train + n_val],
    "test": split_order[n_train + n_val :],
}
print(f"Reloaded {len(unit_frames)} simulator CSVs from {DATA_DIR}")
unit_summary

In [ ]:
data_trace_path = save_unit_data_trace(
    unit_frames,
    unit_names_by_split,
    ASSET_DIR / "picid_data_trace.png",
)
display(Image(filename=str(data_trace_path)))

## 2. Compose the PICID datasource

The simulator CSVs are read from disk into pandas DataFrames because PICID's `load_from_csv()` is its CSV/DataFrame-mode constructor rather than a path reader. The named child loaders then expose complete lifetimes. One `BySourceSplitter` on the multi-source datasource assigns 18 complete reactors to training, six to validation, and six to test. No reactor—and therefore no window from that reactor—appears in more than one split.

In [ ]:
unit_loaders = {
    unit_name: CustomSingleSourceLoader.load_from_csv(
        source=frame,
        target_column="rul",
        task_mode="rul",
        data_name=unit_name,
        is_part_of_multisource=True,
    )
    for unit_name, frame in unit_frames.items()
}
datasource = CustomMultiSourceLoader(
    sources=unit_loaders,
    task_mode="rul",
    data_name="by_unit_ragged_units",
    multisource_data_splitter=BySourceSplitter(
        sources_train=unit_names_by_split["train"],
        sources_val=unit_names_by_split["val"],
        sources_test=unit_names_by_split["test"],
    ),
)
unit_names_by_split

## 3. Build three model-specific preprocessing pipelines

These are the central preprocessing calls. The deep-learning view retains per-unit `(time, feature)` arrays. The pointwise fit/predict view adds only a leading task dimension. The history fit/predict view uses PICID's `TimeseriesTabularizer`: each row contains 24 complete timesteps (`24 × 6 = 144` inputs), and its target is the RUL at the final timestep. Windows are created within units before PICID concatenates them, so no history crosses a reactor boundary.

In [ ]:
experiment = PicidExperiment()

deep_learning_processed = experiment.process_datasource(
    datasource,
    transforms=scaling_transforms(),
    cache=False,
)

In [ ]:
windowed_fit_predict_processed = experiment.process_datasource(
    datasource,
    transforms=[
        *scaling_transforms(),
        DataTransform(
            transform_name="tabularize_24_step_features",
            transform=TimeseriesTabularizer(
                select_features=OmegaConf.create([{"features": "history"}]),
                timestep_dimension=1,
                seq_len=SEQ_LEN,
                label_len=0,
                pred_len=0,
                stride=STRIDE,
                padding_left_flag=False,
            ),
            metadata={"apply_to": ["features"], "assign_to": "features"},
        ),
        DataTransform(
            transform_name="tabularize_present_rul",
            transform=TimeseriesTabularizer(
                select_features=OmegaConf.create([{"rul": "present"}]),
                timestep_dimension=1,
                seq_len=SEQ_LEN,
                label_len=0,
                pred_len=0,
                stride=STRIDE,
                padding_left_flag=False,
            ),
            metadata={"apply_to": ["rul"], "assign_to": "rul"},
        ),
    ],
    cache=False,
)

In [ ]:
fit_predict_processed = experiment.process_datasource(
    datasource,
    transforms=[
        *scaling_transforms(),
        DataTransform(
            transform_name="add_task_dim_features",
            transform=ReshapeTransform("time feature -> 1 time feature"),
            metadata={"apply_to": "features", "assign_to": "features"},
        ),
        DataTransform(
            transform_name="add_task_dim_rul",
            transform=ReshapeTransform("time target -> 1 time target"),
            metadata={"apply_to": "rul", "assign_to": "rul"},
        ),
    ],
    cache=False,
)

In [ ]:
prepared = summarize_prepared_experiment(
    experiment=experiment,
    unit_frames=unit_frames,
    unit_summary=unit_summary,
    unit_names=unit_names,
    unit_names_by_split=unit_names_by_split,
    deep_learning_processed=deep_learning_processed,
    fit_predict_processed=fit_predict_processed,
    windowed_fit_predict_processed=windowed_fit_predict_processed,
    seq_len=SEQ_LEN,
)

for split in ("train", "val", "test"):
    lengths = prepared.unit_lengths_by_split[split]
    print(
        f"{split:>5}: units={prepared.unit_names_by_split[split]} | "
        f"lengths={lengths} | ragged={len(set(lengths.values())) > 1}"
    )
print(f"deep-learning train shapes: {prepared.deep_shapes}")
print(f"pointwise fit/predict shapes: {prepared.fit_predict_shapes}")
print(f"24-step fit/predict shapes:   {prepared.windowed_fit_predict_shapes}")
Console(width=180).print(split_alignment_table(prepared))

## 4. Implement a custom 1D U-Net

The backbone is deliberately defined here: a shallow encoder–bottleneck–decoder with one skip connection, global pooling, and one RUL output per input window. `CustomModelTrainer` adapts it to PICID's batch dictionary and regression output contract.

In [ ]:
class ConvBlock1D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(1, out_channels),
            nn.ReLU(),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(1, out_channels),
            nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


class SmallUNet1D(nn.Module):
    """Small U-Net regressor for PICID windows shaped (batch, time, feature)."""

    def __init__(self, input_channels: int, base_channels: int = 8) -> None:
        super().__init__()
        self.encoder = ConvBlock1D(input_channels, base_channels)
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.bottleneck = ConvBlock1D(base_channels, 2 * base_channels)
        self.up = nn.ConvTranspose1d(
            2 * base_channels, base_channels, kernel_size=2, stride=2
        )
        self.decoder = ConvBlock1D(2 * base_channels, base_channels)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(base_channels, 1)

    def forward(
        self,
        x: torch.Tensor,
        batch: dict[str, torch.Tensor] | None = None,
        **kwargs,
    ) -> torch.Tensor:
        del batch, kwargs
        x = x.transpose(1, 2)
        skip = self.encoder(x)
        latent = self.bottleneck(self.pool(skip))
        decoded = self.up(latent)
        if decoded.shape[-1] != skip.shape[-1]:
            decoded = torch.nn.functional.interpolate(
                decoded, size=skip.shape[-1], mode="linear", align_corners=False
            )
        decoded = self.decoder(torch.cat([skip, decoded], dim=1))
        return self.head(self.global_pool(decoded).squeeze(-1))


torch.manual_seed(SEED)
custom_unet_trainer = CustomModelTrainer(
    task_type="rul",
    model=SmallUNet1D(input_channels=N_FEATURES, base_channels=8),
)

with torch.no_grad():
    shape_check = custom_unet_trainer.backbone(
        torch.zeros(2, SEQ_LEN, N_FEATURES)
    ).shape
assert shape_check == (2, 1)
print(f"Custom backbone output shape: {shape_check}")

## 5. Defaults

The seven branches share one task definition, logger setup, runtime settings, and two families of trainer overrides. Fit/predict wrapper models also share one Hydra defaults list. Keeping those defaults here makes every experiment cell below describe only what differs.

In [ ]:
task_definition = Prognostic(
    task_type="rul",
    seq_len=SEQ_LEN,
    stride=STRIDE,
    stride_train=STRIDE,
)

COMMON_TRAIN_DEFAULTS = {
    "task_definition": task_definition,
    "transforms": [],
    "evaluators": "default",
    "enable_progress_bar": False,
    "seed": SEED,
}
COMMON_RUNTIME_OVERRIDES = [
    "trainer.accelerator=cpu",
    "trainer.devices=1",
    "datamodule.num_workers=0",
]
FIT_PREDICT_TRAIN_DEFAULTS = {
    "train_kwargs": COMMON_TRAIN_DEFAULTS,
    "overrides": ["trainer.max_epochs=1", *COMMON_RUNTIME_OVERRIDES],
}
DEEP_LEARNING_TRAIN_DEFAULTS = {
    "train_kwargs": COMMON_TRAIN_DEFAULTS,
    "overrides": [
        f"trainer.max_epochs={DEEP_LEARNING_EPOCHS}",
        *COMMON_RUNTIME_OVERRIDES,
        "+trainer.log_every_n_steps=1",
        (
            "+callbacks.resource_tracker._target_="
            "picid.callbacks.resource_tracker.ResourceTracker"
        ),
        "+callbacks.resource_tracker.skip_first_n=0",
        "+callbacks.model_checkpoint.enable_version_counter=False",
    ],
}
FIT_PREDICT_MODEL_DEFAULTS = [
    {"override /dataset": "fit_predict_task"},
    {"override /datamodule": "base_datamodule_fit_predict"},
    {"override /callbacks": "gradient_free"},
    {"override /trainer": "one_epoch"},
]


def fit_predict_model_config(target, cache_name, **model_kwargs):
    return {
        "defaults": [*FIT_PREDICT_MODEL_DEFAULTS],
        "task_definition": {"requires_training": True},
        "model": {
            "_target_": target,
            "task_type": "${task_definition.task_type}",
            "model_cache_path": f"${{paths.model_cache_dir}}/{cache_name}",
            **model_kwargs,
        },
    }


def train_branch(*, run_name, model, datasource, defaults, overrides=()):
    return experiment.train(
        run_name=run_name,
        model=model,
        datasource=datasource,
        loggers=[CsvLogger(name=run_name, version=TRACE_VERSION)],
        overrides=[*defaults["overrides"], *overrides],
        **defaults["train_kwargs"],
    )

## 6. Train all seven branches

XGBoost and Bayesian ridge are each run twice through PICID's fit/predict execution path: once from the current six measurements, and once from the 144-column history produced by `TimeseriesTabularizer`. A regularized scikit-learn decision tree adds a third history-aware fit/predict baseline. All history-aware paths share the neural branches' 24-sample receptive field and stride of four. The PICID MLP and custom U-Net have a 30-epoch maximum with per-step logging and `ResourceTracker`; validation-based early stopping may end a neural branch sooner.

In [ ]:
Console().print(execution_comparison_table())

In [ ]:
fit_predict_results = train_branch(
    run_name="picid_experiment_fit_predict",
    model="xgboost_fit_predict",
    datasource=fit_predict_processed,
    defaults=FIT_PREDICT_TRAIN_DEFAULTS,
    overrides=[
        f"+model.n_estimators={N_ESTIMATORS}",
        (
            "+model.model_cache_path="
            f"{WORKSHOP_ROOT / 'artifacts' / 'model_cache' / 'xgboost'}"
        ),
    ],
)

In [ ]:
bayesian_fit_predict_results = train_branch(
    run_name="picid_experiment_bayesian_ridge",
    model=fit_predict_model_config(
        target="picid_training.FitPredictBayesianRidgeWrapper",
        cache_name="bayesian_ridge",
        compute_score=True,
    ),
    datasource=fit_predict_processed,
    defaults=FIT_PREDICT_TRAIN_DEFAULTS,
)

In [ ]:
windowed_fit_predict_results = train_branch(
    run_name="picid_experiment_fit_predict_24_step",
    model="xgboost_fit_predict",
    datasource=windowed_fit_predict_processed,
    defaults=FIT_PREDICT_TRAIN_DEFAULTS,
    overrides=[
        f"+model.n_estimators={N_ESTIMATORS}",
        (
            "+model.model_cache_path="
            f"{WORKSHOP_ROOT / 'artifacts' / 'model_cache' / 'xgboost_24_step'}"
        ),
    ],
)

In [ ]:
bayesian_windowed_fit_predict_results = train_branch(
    run_name="picid_experiment_bayesian_ridge_24_step",
    model=fit_predict_model_config(
        target="picid_training.FitPredictBayesianRidgeWrapper",
        cache_name="bayesian_ridge_24_step",
        compute_score=True,
    ),
    datasource=windowed_fit_predict_processed,
    defaults=FIT_PREDICT_TRAIN_DEFAULTS,
)

In [ ]:
decision_tree_results = train_branch(
    run_name="picid_experiment_decision_tree_24_step",
    model=fit_predict_model_config(
        target="picid_training.FitPredictDecisionTreeWrapper",
        cache_name="decision_tree_24_step",
        max_depth=6,
        min_samples_leaf=20,
        random_state=SEED,
    ),
    datasource=windowed_fit_predict_processed,
    defaults=FIT_PREDICT_TRAIN_DEFAULTS,
)

In [ ]:
deep_learning_results = train_branch(
    run_name="picid_experiment_deep_learning",
    model="mlp",
    datasource=deep_learning_processed,
    defaults=DEEP_LEARNING_TRAIN_DEFAULTS,
    overrides=[
        f"callbacks.model_checkpoint.dirpath={CHECKPOINT_DIR / 'mlp'}",
    ],
)

In [ ]:
custom_unet_results = train_branch(
    run_name="picid_experiment_custom_unet",
    model=custom_unet_trainer,
    datasource=deep_learning_processed,
    defaults=DEEP_LEARNING_TRAIN_DEFAULTS,
    overrides=[
        f"callbacks.model_checkpoint.dirpath={CHECKPOINT_DIR / 'custom_unet'}",
    ],
)

In [ ]:
run = package_training_run(
    fit_predict_results=fit_predict_results,
    bayesian_fit_predict_results=bayesian_fit_predict_results,
    windowed_fit_predict_results=windowed_fit_predict_results,
    bayesian_windowed_fit_predict_results=(
        bayesian_windowed_fit_predict_results
    ),
    decision_tree_results=decision_tree_results,
    deep_learning_results=deep_learning_results,
    custom_unet_results=custom_unet_results,
    seed=SEED,
    deep_learning_epochs=DEEP_LEARNING_EPOCHS,
)
run.result_frame

## 7. Unroll all persisted models across complete lifetimes

PICID's default evaluator returns aggregate errors. For a stronger sanity check, restore the five fit/predict estimators and the selected MLP/U-Net checkpoints, then place every prediction at the common 24-step window endpoints. The two pointwise models are sampled at those endpoints; the tabularized and neural models consume the complete preceding window. The comparison baseline is one constant: the mean target over training windows only. Positive skill means lower MSE than that constant.

In [ ]:
checkpoint_unroll = build_checkpoint_unroll(
    prepared=prepared,
    run=run,
    custom_unet_model=custom_unet_trainer.backbone,
    stride=task_definition.stride,
)

for model_name in (
    "XGBoost (24-step)",
    "Bayesian ridge (24-step)",
    "Decision tree (24-step)",
    "MLP",
    "custom U-Net",
):
    checkpoint_mse = checkpoint_unroll.split_summary.loc[
        (checkpoint_unroll.split_summary["split"] == "test")
        & (checkpoint_unroll.split_summary["model"] == model_name),
        "mse",
    ].iloc[0]
    returned_mse = run.result_frame.loc[
        run.result_frame["family"] == model_name,
        "test/mse_normalized",
    ].iloc[0]
    assert abs(checkpoint_mse - returned_mse) < 1e-6

print(f"MLP checkpoint:   {run.mlp_checkpoint_path.relative_to(WORKSHOP_ROOT)}")
print(
    "U-Net checkpoint: "
    f"{run.custom_unet_checkpoint_path.relative_to(WORKSHOP_ROOT)}"
)
print(
    "24-step caches:  "
    f"{run.windowed_xgboost_model_path.relative_to(WORKSHOP_ROOT)}, "
    f"{run.windowed_bayesian_model_path.relative_to(WORKSHOP_ROOT)}"
)
print(
    "Decision tree:   "
    f"{run.decision_tree_model_path.relative_to(WORKSHOP_ROOT)}"
)
checkpoint_unroll.split_summary

## 8. Persist and inspect the run traces

The plotting helper consumes the objects created above. The PICID evaluator remains aggregate because the datasource has no `unit_id` metadata; the persisted-model unrolls recover unit alignment from the explicit by-unit lists retained by this notebook.

In [ ]:
trace_paths = save_run_traces(
    prepared,
    run,
    checkpoint_unroll,
    asset_dir=ASSET_DIR,
    artifact_dir=ARTIFACT_DIR,
    seed=SEED,
)

for trace_name, trace_path in trace_paths.items():
    print(f"{trace_name:>12}: {trace_path.relative_to(WORKSHOP_ROOT)}")
    display(Image(filename=str(trace_path)))

## Interpretation

The returned numbers validate three input/model execution paths on a synthetic protocol; they are not an algorithm ranking. The lifetime unrolls test whether every persisted model varies meaningfully over a lifetime and beats a training-only constant. The pointwise-versus-24-step pairs isolate the effect of adding temporal context while holding the estimator family fixed; the regularized decision tree provides an interpretable nonlinear single-tree baseline. For a scientific comparison, replace the generated frames with a domain dataset, tune all branches, and add aligned unit metadata for native per-unit evaluator traces.